In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [19]:
# 1. Charger les données

def load_words(path="names.txt"):
    with open(path, "r", encoding="utf-8") as f:
        return f.read().splitlines()

w = load_words()
words = w[:30]

In [20]:
# 2. Vocabulaire

chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(stoi)

In [21]:
vocab_size

22

In [22]:
# 3. Dataset (contexte = 3)

block_size = 3
X, Y = [], []

for w in words:
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)

In [23]:
X

tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 11],
        [ 5, 11, 11],
        [11, 11,  1],
        [ 0,  0,  0],
        [ 0,  0, 13],
        [ 0, 13, 10],
        [13, 10,  9],
        [10,  9, 19],
        [ 9, 19,  9],
        [19,  9,  1],
        [ 0,  0,  0],
        [ 0,  0,  1],
        [ 0,  1, 19],
        [ 1, 19,  1],
        [ 0,  0,  0],
        [ 0,  0,  9],
        [ 0,  9, 16],
        [ 9, 16,  1],
        [16,  1,  2],
        [ 1,  2,  5],
        [ 2,  5, 10],
        [ 5, 10, 10],
        [10, 10,  1],
        [ 0,  0,  0],
        [ 0,  0, 16],
        [ 0, 16, 13],
        [16, 13, 14],
        [13, 14,  8],
        [14,  8,  9],
        [ 8,  9,  1],
        [ 0,  0,  0],
        [ 0,  0,  3],
        [ 0,  3,  8],
        [ 3,  8,  1],
        [ 8,  1, 15],
        [ 1, 15, 10],
        [15, 10, 13],
        [10, 13, 17],
        [13, 17, 17],
        [17, 17,  5],
        [ 0,  0,  0],
        [ 0,  0, 11],
        [ 0, 11,  9],
        [1

In [24]:
Y

tensor([ 5, 11, 11,  1,  0, 13, 10,  9, 19,  9,  1,  0,  1, 19,  1,  0,  9, 16,
         1,  2,  5, 10, 10,  1,  0, 16, 13, 14,  8,  9,  1,  0,  3,  8,  1, 15,
        10, 13, 17, 17,  5,  0, 11,  9,  1,  0,  1, 11,  5, 10,  9,  1,  0,  8,
         1, 15, 14,  5, 15,  0,  5, 19,  5, 10, 20, 12,  0,  1,  2,  9,  7,  1,
         9, 10,  0,  5, 11,  9, 10, 20,  0,  5, 10,  9, 21,  1,  2,  5, 17,  8,
         0, 11,  9, 10,  1,  0,  5, 10, 10,  1,  0,  1, 19,  5, 15, 20,  0, 16,
        13,  6,  9,  1,  0,  3,  1, 11,  9, 10,  1,  0,  1, 15,  9,  1,  0, 16,
         3,  1, 15, 10,  5, 17, 17,  0, 19,  9,  3, 17, 13, 15,  9,  1,  0, 11,
         1,  4,  9, 16, 13, 12,  0, 10, 18, 12,  1,  0,  7, 15,  1,  3,  5,  0,
         3,  8, 10, 13,  5,  0, 14,  5, 12,  5, 10, 13, 14,  5,  0, 10,  1, 20,
        10,  1,  0, 15,  9, 10,  5, 20,  0, 21, 13,  5, 20,  0, 12, 13, 15,  1,
         0])

In [25]:
# 4. Modèle MLP

class MakemoreMLP(nn.Module):
    def __init__(self, vocab_size, block_size, n_embd=10, n_hidden=200):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, n_embd)
        self.fc1 = nn.Linear(block_size * n_embd, n_hidden)
        self.fc2 = nn.Linear(n_hidden, vocab_size)

    def forward(self, x):
        emb = self.embedding(x)
        x = emb.view(emb.shape[0], -1)
        x = torch.tanh(self.fc1(x))
        logits = self.fc2(x)
        return logits

model = MakemoreMLP(vocab_size, block_size)

In [26]:
# 5. Entraînement

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for step in range(20000):
    logits = model(X)
    loss = F.cross_entropy(logits, Y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 2000 == 0:
        print(f"step {step}: loss = {loss.item():.4f}")

step 0: loss = 3.0617
step 2000: loss = 0.5504
step 4000: loss = 0.5510
step 6000: loss = 0.5503
step 8000: loss = 0.5504
step 10000: loss = 0.5507
step 12000: loss = 0.5504
step 14000: loss = 0.5502
step 16000: loss = 0.5502
step 18000: loss = 0.5503


In [27]:
# 6. Génération

@torch.no_grad()
def generate(model, max_length=20):
    out = []
    context = [0] * block_size

    for _ in range(max_length):
        x = torch.tensor([context])
        logits = model(x)
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1).item()

        if ix == 0:
            break

        out.append(itos[ix])
        context = context[1:] + [ix]

    return ''.join(out)

# 7. Test

for _ in range(20):
    print(generate(model))

ava
zoey
elia
amelizabeth
madison
evelyn
charlotte
sofia
madison
olivia
amelizabeth
evelyn
grace
sophia
emila
abigail
luna
riley
isabeth
harlett


In [28]:
# Version Bigram simple

def train_bigram(words):
    N = torch.zeros((vocab_size, vocab_size))

    for w in words:
        chs = ['.'] + list(w) + ['.']
        for ch1, ch2 in zip(chs, chs[1:]):
            i = stoi[ch1]
            j = stoi[ch2]
            N[i, j] += 1

    P = (N + 1)
    P /= P.sum(1, keepdim=True)
    return P

P = train_bigram(words)

@torch.no_grad()
def generate_bigram(P):
    out = []
    ix = 0
    while True:
        p = P[ix]
        ix = torch.multinomial(p, num_samples=1).item()
        if ix == 0:
            break
        out.append(itos[ix])
    return ''.join(out)

for _ in range(10):
    print(generate_bigram(P))

eyoffegaararfivvzhbay
evlbe
mufresthvaoedlaiica
arhoiymmiloimd
erm
adgnluhavtbeuza
oduduasotlngtonoflotccpcbnbcbmcmsn
ea
ccmpmsyacicil
a


In [29]:
# LSTM

In [30]:
block_size = 8

def build_dataset(words):
    X, Y = [], []

    for w in words:
        chs = ['.'] + list(w) + ['.']
        for i in range(len(chs)-1):
            context = chs[:i+1]
            target = chs[i+1]

            context = context[-block_size:]
            context = ['.'] * (block_size - len(context)) + context

            X.append([stoi[c] for c in context])
            Y.append(stoi[target])

    return torch.tensor(X), torch.tensor(Y)

X, Y = build_dataset(words)

In [31]:
import torch.nn as nn

class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.linear = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        emb = self.embedding(x)
        out, _ = self.lstm(emb)
        logits = self.linear(out)
        return logits
    
model = CharLSTM(vocab_size, embed_dim=16, hidden_dim=128)

In [32]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

for step in range(2000):
    logits = model(X)

    logits_last = logits[:, -1, :]

    loss = torch.nn.functional.cross_entropy(logits_last, Y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 200 == 0:
        print(loss.item())

3.1190571784973145
0.5234354138374329
0.5153601765632629
0.5141144394874573
0.5134723782539368
0.5132339596748352
0.5131649971008301
0.513006865978241
0.5129552483558655
0.5129116773605347


In [33]:
@torch.no_grad()
def generate_lstm(model):
    out = []
    context = [0] * block_size

    while True:
        x = torch.tensor([context])
        logits = model(x)
        probs = torch.softmax(logits[:, -1, :], dim=1)

        ix = torch.multinomial(probs, 1).item()

        if ix == 0:
            break

        out.append(itos[ix])
        context = context[1:] + [ix]

    return ''.join(out)

for _ in range(20):
    print(generate_lstm(model))

scarlett
elizabeth
evelyn
penelope
emily
mia
penelope
luna
emily
mia
amelia
penelope
elizabeth
charlotte
zoey
victoria
riley
luna
isabella
sophia


In [34]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_embd=64, n_head=4, n_layer=2):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=n_embd,
            nhead=n_head,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layer)

        self.ln = nn.LayerNorm(n_embd)
        self.fc = nn.Linear(n_embd, vocab_size)

        self.block_size = block_size

    def forward(self, x):
        B, T = x.shape

        tok_emb = self.token_embedding(x)
        pos = torch.arange(T)
        pos_emb = self.position_embedding(pos)

        x = tok_emb + pos_emb

        # masque causal (important !)
        mask = torch.triu(torch.ones(T, T), diagonal=1).bool()
        x = self.transformer(x, mask=mask)

        x = self.ln(x)
        logits = self.fc(x)

        return logits

In [35]:
model = MiniGPT(vocab_size, block_size)
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

In [36]:
for step in range(3000):
    logits = model(X)

    logits_last = logits[:, -1, :]
    loss = F.cross_entropy(logits_last, Y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 300 == 0:
        print(loss.item())

3.2858128547668457
0.547619640827179
0.5483570694923401
0.5276796221733093
0.5300606489181519
0.5523058772087097
0.5359213948249817
0.5119979977607727
0.5211287140846252
0.5186083316802979


In [37]:
@torch.no_grad()
def generate_transformer(model):
    out = []
    context = [0] * block_size

    while True:
        x = torch.tensor([context])
        logits = model(x)

        probs = F.softmax(logits[:, -1, :], dim=1)
        ix = torch.multinomial(probs, 1).item()

        if ix == 0:
            break

        out.append(itos[ix])
        context = context[1:] + [ix]

    return ''.join(out)

In [38]:
for _ in range(20):
    print(generate_transformer(model))

abigail
avery
isabella
charlotte
nora
luna
layla
evelyn
nora
zoey
amelia
aria
camila
penelope
layla
abigail
charlotte
olivia
evelyn
camila
